## State level descriptive overview

A first look at Mississippi across the joined datasets, covering how big the state is, where the most vulnerable tracts sit, the Delta diabetes story, and the five year trends.

### Importing necessary libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv('../.env')
host = os.getenv('PGHOST', 'localhost')
port = os.getenv('PGPORT', '5432')
db = os.getenv('PGDATABASE', 'ms_health')
user = os.getenv('PGUSER', 'postgres')
pwd = os.getenv('PGPASSWORD', '')
engine = create_engine(f'postgresql+psycopg://{user}:{pwd}@{host}:{port}/{db}')
sns.set(style='whitegrid')

def q(sql):
    return pd.read_sql(text(sql), engine)

### State level snapshot

In [2]:
# Headline counts for project scale.
q("""
SELECT 'MS census tracts' AS metric, COUNT(*)::TEXT AS value FROM dim_geography
UNION ALL SELECT 'MS counties', COUNT(DISTINCT county_fips)::TEXT FROM dim_geography
UNION ALL SELECT 'Total population', TO_CHAR(SUM(total_population), 'FM999,999,999') FROM dim_geography
UNION ALL SELECT 'PLACES tract-measures', TO_CHAR(COUNT(*), 'FM999,999,999') FROM fact_places
UNION ALL SELECT 'PLACES years', TO_CHAR(COUNT(DISTINCT year_sk), '9') FROM fact_places
UNION ALL SELECT 'MS hospitals', COUNT(*)::TEXT FROM dim_facility WHERE state_abbr='MS'
UNION ALL SELECT 'Birthing-friendly hospitals (MS)',
    COUNT(*)::TEXT FROM dim_facility WHERE state_abbr='MS' AND is_birthing_friendly
""")

,metric,value
0,Birthing-friendly hospitals (MS),37
1,MS hospitals,108
2,Total population,"2,958,846"
3,MS census tracts,878
4,MS counties,82
5,PLACES years,7
6,PLACES tract-measures,"74,201"


### Top 20 most vulnerable tracts

In [3]:
# Demonstrates four different window functions in one query: ROW_NUMBER for
# top-N filtering, DENSE_RANK to handle ties, PERCENT_RANK for percentile,
# and NTILE(5) to bucket tracts into quintiles.
q("""
WITH ranked AS (
    SELECT g.tract_fips, g.county_name, g.region, g.total_population,
        s.rpl_themes,
        ROW_NUMBER() OVER (ORDER BY s.rpl_themes DESC NULLS LAST) AS rn,
        DENSE_RANK() OVER (ORDER BY s.rpl_themes DESC NULLS LAST) AS svi_rank,
        PERCENT_RANK() OVER (ORDER BY s.rpl_themes) AS svi_pctile,
        NTILE(5) OVER (ORDER BY s.rpl_themes) AS svi_quintile
    FROM dim_geography g
    JOIN fact_svi_wide s ON s.geo_sk = g.geo_sk
)
SELECT rn, tract_fips, county_name, region, total_population,
    ROUND(rpl_themes::NUMERIC, 3) AS svi_overall,
    svi_rank, svi_quintile
FROM ranked
WHERE rn <= 20
ORDER BY rn
""")

,rn,tract_fips,county_name,region,total_population,svi_overall,svi_rank,svi_quintile
0,1,28151001000,Washington,Delta,1220,1.000,1,5
1,2,28153950200,Wayne,Other,4644,0.999,2,5
2,3,28011950200,Bolivar,Delta,2736,0.998,3,5
3,4,28123020400,Scott,Other,3251,0.997,4,5
4,5,28107950201,Panola,Delta,1776,0.995,5,5
5,6,28073020305,Lamar,Pine Belt,3687,0.994,6,5
6,7,28151000400,Washington,Delta,2589,0.993,7,5
7,8,28035000601,Forrest,Pine Belt,2607,0.992,8,5
8,9,28075000600,Lauderdale,Other,1550,0.991,9,5
9,10,28059042201,Jackson,Gulf Coast,2081,0.990,10,5


### Diabetes Belt: Delta vs the rest of Mississippi

In [4]:
# Pop-weighted instead of plain average - bigger tracts contribute more.
# A simple AVG would treat a 500-person tract the same as a 5000-person tract.
q("""
WITH places_diabetes AS (
    SELECT g.is_delta, g.total_population, f.data_value AS diabetes_pct
    FROM fact_places f
    JOIN dim_geography g ON g.geo_sk = f.geo_sk
    JOIN dim_measure m ON m.measure_sk = f.measure_sk
    WHERE m.source = 'PLACES' AND m.measure_id = 'DIABETES'
    AND f.year_sk = (SELECT MAX(year_sk) FROM fact_places)
)
SELECT
    CASE WHEN is_delta THEN 'Delta (18 counties)' ELSE 'Rest of MS' END AS region,
    COUNT(*) AS tracts,
    SUM(total_population) AS population,
    ROUND(SUM(diabetes_pct * total_population) / NULLIF(SUM(total_population), 0)::NUMERIC, 2) AS pop_weighted_diabetes_pct,
    ROUND(MIN(diabetes_pct)::NUMERIC, 1) AS min_pct,
    ROUND(MAX(diabetes_pct)::NUMERIC, 1) AS max_pct
FROM places_diabetes
GROUP BY is_delta
ORDER BY is_delta DESC
""")

,region,tracts,population,pop_weighted_diabetes_pct,min_pct,max_pct
0,Delta (18 counties),153,537726,17.24,8.8,31.7
1,Rest of MS,717,2421015,15.47,2.6,32.6


Delta sits noticeably higher than the rest of the state, and both are well above the national average of roughly 11 percent.

### Five year trends by region

In [5]:
# Pivot 8 health measures into columns using FILTER, broken out by region and year.
# This is the multi-year companion to the Delta-vs-rest snapshot above.
trends = q("""
SELECT f.year_sk AS year, g.region,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'DIABETES')::NUMERIC, 2) AS diabetes,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'OBESITY')::NUMERIC, 2) AS obesity,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'CSMOKING')::NUMERIC, 2) AS smoking,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'BPHIGH')::NUMERIC, 2) AS bp_high,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'DEPRESSION')::NUMERIC, 2) AS depression,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'MHLTH')::NUMERIC, 2) AS mental_distress,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'CHECKUP')::NUMERIC, 2) AS routine_checkup,
    ROUND(AVG(f.data_value) FILTER (WHERE m.measure_id = 'ACCESS2')::NUMERIC, 2) AS uninsured
FROM fact_places f
JOIN dim_geography g ON g.geo_sk = f.geo_sk
JOIN dim_measure m ON m.measure_sk = f.measure_sk
WHERE m.source = 'PLACES'
AND m.measure_id IN ('DIABETES','OBESITY','CSMOKING','BPHIGH','DEPRESSION','MHLTH','CHECKUP','ACCESS2')
GROUP BY f.year_sk, g.region
ORDER BY f.year_sk, g.region
""")
trends

,year,region,diabetes,obesity,smoking,bp_high,depression,mental_distress,routine_checkup,uninsured
0,2017,Delta,NaN,NaN,NaN,44.12,NaN,NaN,NaN,NaN
1,2017,Gulf Coast,NaN,NaN,NaN,40.39,NaN,NaN,NaN,NaN
2,2017,Jackson Metro,NaN,NaN,NaN,40.14,NaN,NaN,NaN,NaN
3,2017,Other,NaN,NaN,NaN,43.50,NaN,NaN,NaN,NaN
4,2017,Pine Belt,NaN,NaN,NaN,39.25,NaN,NaN,NaN,NaN
5,2018,Delta,16.54,42.61,24.73,NaN,NaN,18.27,77.69,22.81
6,2018,Gulf Coast,13.73,37.92,21.36,NaN,NaN,16.16,76.60,19.62
7,2018,Jackson Metro,13.73,41.97,20.27,NaN,NaN,15.74,78.23,18.70
8,2018,Other,15.66,40.75,24.10,NaN,NaN,17.55,76.73,21.11
9,2018,Pine Belt,13.59,41.69,22.79,NaN,NaN,17.42,75.86,20.61
